In [1]:
import pandas as pd
import os
import glob
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 统一配置接口
# ==========================================
INPUT_DIR = "./engineered_features"
OUTPUT_FILE = "P4_Cleaned_Dataset.csv"

# 黑名单：强制剔除的特征名称
EXCLUDE_FEATURES = [
    'Cum_GDD_P1',
    'Cum_GDD_P2',
    'Cum_GDD_P3',
    'Cum_GDD_P4',
    'Cum_GDD_P5'
]

# 白名单：需要保留的静态特征与元数据列
STATIC_FEATURES = ['Sand', 'Clay', 'SOC']
META_COLS = ['Year', 'Zone', 'latitude', 'longitude', 'yield']

# 目标物候期（P4 阶段包含 P1 至 P4 的所有信息）
ACTIVE_PERIODS = ['P1', 'P2', 'P3', 'P4']
# ==========================================

def prepare_p4_dataset(input_dir, output_file):
    print(f"正在读取 {input_dir} 目录下的所有 CSV 文件...")
    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    if not csv_files:
        print("错误：未找到 CSV 文件，请检查路径。")
        return
        
    # 读取并拼接所有数据
    df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    all_columns = df.columns.tolist()
    
    # 动态特征筛选逻辑
    dynamic_features = [
        col for col in all_columns 
        if any(p in col for p in ACTIVE_PERIODS)  # 包含 P1, P2, P3, P4
        and 'P5' not in col                       # 严格排除 P5 的未来特征
        and col not in META_COLS                  # 排除元数据（防止重复）
        and col not in STATIC_FEATURES            # 排除静态特征（防止重复）
        and col not in EXCLUDE_FEATURES           # 执行黑名单剔除
    ]
    
    # 组装最终需要保留的所有列
    final_columns = META_COLS + STATIC_FEATURES + dynamic_features
    
    # 确保所有列都在数据框中（防止拼写错误或列不存在）
    final_columns = [col for col in final_columns if col in df.columns]
    
    # 切片生成清洗后的数据集
    cleaned_df = df[final_columns]
    
    # 去除可能包含空值的行（确保 XGBoost 和 SHAP 分析不报错）
    original_len = len(cleaned_df)
    cleaned_df = cleaned_df.dropna()
    dropped_len = original_len - len(cleaned_df)
    
    # 导出文件
    cleaned_df.to_csv(output_file, index=False)
    
    # 打印核对信息
    print(f"\n{'='*50}")
    print(f"P4 阶段数据集处理完成！")
    print(f"{'='*50}")
    print(f"总样本数: {len(cleaned_df)} (剔除了 {dropped_len} 个含缺失值的样本)")
    print(f"特征总数 (含元数据): {len(final_columns)}")
    print(f"  - 元数据列: {len(META_COLS)}个 {META_COLS}")
    print(f"  - 静态特征: {len(STATIC_FEATURES)}个 {STATIC_FEATURES}")
    print(f"  - 动态特征: {len(dynamic_features)}个 (P1-P4)")
    print(f"已强制排除的特征: {EXCLUDE_FEATURES}")
    print(f"文件已保存至: {output_file}")

if __name__ == "__main__":
    prepare_p4_dataset(INPUT_DIR, OUTPUT_FILE)

正在读取 ./engineered_features 目录下的所有 CSV 文件...

P4 阶段数据集处理完成！
总样本数: 119385 (剔除了 0 个含缺失值的样本)
特征总数 (含元数据): 105
  - 元数据列: 5个 ['Year', 'Zone', 'latitude', 'longitude', 'yield']
  - 静态特征: 3个 ['Sand', 'Clay', 'SOC']
  - 动态特征: 97个 (P1-P4)
已强制排除的特征: ['Cum_GDD_P1', 'Cum_GDD_P2', 'Cum_GDD_P3', 'Cum_GDD_P4', 'Cum_GDD_P5']
文件已保存至: P4_Cleaned_Dataset.csv
